# Morphoné Model via HuBERT Soft - Training & Inference Pipeline
This notebook handles dataset preparation via interactive widgets, preprocessing, model training, and inference for the DDSP-SVC model using the HuBERT Soft encoder.

**Reference:** DDSP-SVC framework - https://github.com/yxlllc/DDSP-SVC

## Cell 1 — Environment Setup and Dependencies
Clones the repository directly into the workspace, configures the working directory, and installs all required libraries and dependencies.

In [ ]:
import os

# Clone the repository directly into the execution environment
!git clone https://github.com/your_username/repository_name.git

# Move inside the cloned project folder
repo_name = "repository_name"
if os.path.exists(repo_name):
    %cd {repo_name}
    print(f"Current working directory: {os.getcwd()}")
else:
    print(f"Error: Could not find the folder {repo_name}.")

# Download and install dependencies
!pip install -r requirements.txt

## Cell 2 — Instrument Configuration
Provides interactive widgets to choose up to 5 acoustic instruments and set the maximum number of training samples.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

instruments = [
    'brass_acoustic',
    'flute_acoustic',
    'guitar_acoustic',
    'keyboard_acoustic',
    'mallet_acoustic',
    'organ_acoustic',
    'reed_acoustic',
    'string_acoustic',
    'vocal_acoustic'
]

display(HTML("<h3> SELECT THE TOOLS YOU WANT TO TRAIN ON </h3>"))
display(HTML(f"Select up to 5 instruments and set the number of samples."))

# Creation of selection buttons (via ToggleButton)
display(HTML("<br><b>1. Choose acoustic instruments (max 5):</b>"))
instruments_toggles = []
for s in instruments:
    toggle = widgets.ToggleButton(
        value=False,
        description=s,
        disabled=False,
        button_style='',
        tooltip=f'Select {s}'
    )
    instruments_toggles.append(toggle)

grid_box = widgets.GridBox(
    instruments_toggles,
    layout=widgets.Layout(grid_template_columns='repeat(3, 170px)', grid_gap='8px 8px')
)
display(grid_box)

# Slider for samples number
display(HTML("<br><b>2. Choose the maximum number of samples for training:</b>"))
slider_samples = widgets.IntSlider(
    value=2500,
    min=1,
    max=5000,
    step=50,
    description='Number of samples:',
    style={'description_width': 'initial'}
)
display(slider_samples)

# Button of confirm
btn_confirm = widgets.Button(
    description="Confirm Setting",
    button_style='primary',
    icon='check'
)
display(btn_confirm)

output_result = widgets.Output()
display(output_result)

def on_confirm_clicked(b):
    with output_result:
        output_result.clear_output()
        chosen = [t.description for t in instruments_toggles if t.value]
        chosen_samples = slider_samples.value

        if not chosen:
            print("Error: Select at least an instrument")
        elif len(chosen) > 5:
            print("Error: You can select up to 5 instruments")
        else:
            print("Configuration saved")
            print(f"-> Chosen acoustic instruments ({len(chosen)}/5): {samples}")
            print(f"-> Number of samples for instrument: {chosen_samples}")

            # Direct synchronization with variables expected by the copy cell
            global CHOSEN_INSTRUMENTS, MAX_SAMPLES
            CHOSEN_INSTRUMENTS = chosen
            MAX_SAMPLES = chosen_samples

btn_confirm.on_click(on_confirm_clicked)

## Cell 3 — Dataset Synchronization to Local SSD
Scans the repository for the selected instruments, builds matching dataset pairs, and copies them to the high-speed local SSD for fast processing.

In [ ]:
import shutil
import random
import subprocess
import concurrent.futures
from tqdm.notebook import tqdm

try:
    active_instruments = CHOSEN_INSTRUMENTS if 'CHOSEN_INSTRUMENTS' in globals() and CHOSEN_INSTRUMENTS else ["brass_acoustic"]
    max_active_samples = int(MAX_SAMPLES) if 'MAX_SAMPLES' in globals() else 2500
except Exception:
    active_instruments = ["brass_acoustic"]
    max_active_samples = 2500

CHOSEN_INSTRUMENTS = active_instruments
MAX_SAMPLES = max_active_samples

print(f"Scanning via Terminal")
print(f"Syncronized -> Instruments: {CHOSEN_INSTRUMENTS} | Samples target: {MAX_SAMPLES}\n")

# Use relative paths for your repository structure
DRIVE_TRAIN_AUDIO = "./data/train/audio"
DRIVE_VAL_AUDIO = "./data/val/audio"
LOCAL_SSD_BASE = "/content/dataset_temp"

if os.path.exists(LOCAL_SSD_BASE):
    !rm -rf {LOCAL_SSD_BASE}

os.makedirs(os.path.join(LOCAL_SSD_BASE, "train"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_SSD_BASE, "validation"), exist_ok=True)

def generate_couples_ls(src_dir, dest_dir, is_train=True):
    limit_per_instrument = MAX_SAMPLES if is_train else 10
    folder_dest = os.path.join(dest_dir, "audio")
    os.makedirs(folder_dest, exist_ok=True)

    print(f"Reading file repository (via terminal) for: {CHOSEN_INSTRUMENTS}...")
    candidates_per_instrument = { s: [] for s in CHOSEN_INSTRUMENTS }

    if os.path.exists(src_dir):
        process = subprocess.run(['ls', src_dir], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        files = process.stdout.splitlines()

        for name in files:
            if name.endswith('.wav'):
                for s in CHOSEN_INSTRUMENTS:
                    if name.startswith(s + "_"):
                        candidates_per_instrument[s].append((os.path.join(src_dir, name), os.path.join(folder_dest, name)))
                        break

    couples = []
    for s in CHOSEN_INSTRUMENTS:
        instrument_list = candidates_per_instrument[s]
        random.shuffle(instrument_list)
        chosen = instrument_list[:limit_per_instrument]
        couples.extend(chosen)
        print(f" -> Instrument '{s}': avaiable {len(instrument_list)} file, blocked and selected {len(chosen)}.")

    random.shuffle(couples)

    def copy_attempt(path):
        src, dst = path
        try:
            shutil.copy(src, dst)
            return True
        except Exception:
            return False

    desc_label = "Copy on SSD (Train)" if is_train else "Copy on SSD (Val)"
    if couples:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            results = list(tqdm(executor.map(copy_attempt, couples), total=len(couples), desc=desc_label))
        total_copied = sum(1 for r in results if r)
        print(f"Succesfully copied {total_copied} files on {len(couples)} requested.\n")
    else:
        print(f"None file found inside {src_dir}\n")

generate_couples_ls(DRIVE_TRAIN_AUDIO, os.path.join(LOCAL_SSD_BASE, "train"), is_train=True)
generate_couples_ls(DRIVE_VAL_AUDIO, os.path.join(LOCAL_SSD_BASE, "validation"), is_train=False)

print("Local Copy Completed")

## Cell 4 — Data Preprocessing
Extracts pitch (F0) and HuBERT Soft encoder features, generating the necessary training and validation files.

In [ ]:
# Run preprocessing for feature extraction (F0 and encoder features) using 2 parallel workers
!python preprocess.py -c configs/reflow.yaml -j 2

## Cell 5 — Model Training
Starts the DDSP-SVC training process using the configuration defined in the YAML file and saves checkpoints periodically.

In [ ]:
# Start the model training process using the specified configuration file
!python train_reflow.py -c configs/reflow.yaml

## Cell 6 — Model Inference
Allows you to load a trained model checkpoint, select an input audio file, and run inference to generate synthesized audio outputs.

In [ ]:
import glob

# Use current working directory
ddsp_dir = os.getcwd()
os.chdir(ddsp_dir)

ckpt_files = glob.glob(os.path.join(ddsp_dir, "exp/**/*.pt"), recursive=True) + glob.glob(os.path.join(ddsp_dir, "exp/**/*.ckpt"), recursive=True)

text_wav = widgets.Text(value='Rino_solo_1-8.wav', description='File WAV:', style={'description_width': 'initial'})
dropdown_ckpt = widgets.Dropdown(options=ckpt_files, description='Checkpoint:', style={'description_width': 'initial'})
text_output = widgets.Text(value='output_Brass_14000.wav', description='Output name:', style={'description_width': 'initial'})

display(text_wav, dropdown_ckpt, text_output)

btn = widgets.Button(description="Starts Inference", button_style="primary")
out = widgets.Output()

display(btn, out)

def click(b):
    with out:
        out.clear_output()
        wav_path = text_wav.value

        # Check local relative path / inside project directory
        if not os.path.exists(wav_path):
            if os.path.exists(os.path.join(ddsp_dir, wav_path)):
                wav_path = os.path.join(ddsp_dir, wav_path)
            else:
                print(f"Error: File {text_wav.value} not found locally.")

btn.on_click(click)